<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/02_similarity/similarity_threshold_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
data = [
    ("I love machine learning", "I enjoy studying machine learning", "Relevant"),
    ("She works on NLP systems", "She builds natural language processing pipelines", "Relevant"),
    ("Transformers generate embeddings", "Sentence embeddings help semantic search", "Relevant"),
    ("I love machine learning", "The weather is hot today", "Not Relevant"),
    ("He plays football", "Natural language processing uses text data", "Not Relevant"),
    ("Python is a programming language", "Snakes live in forests", "Not Relevant"),
]

In [7]:
rows = []

for a, b, label in data:
    emb_a = model.encode(a)
    emb_b = model.encode(b)
    score = cosine_similarity([emb_a], [emb_b])[0][0]

    rows.append({
        "Sentence A": a,
        "Sentence B": b,
        "Expected": label,
        "Similarity": round(score, 4)
    })

df = pd.DataFrame(rows)
df

,Sentence A,Sentence B,Expected,Similarity
0,I love machine learning,I enjoy studying machine learning,Relevant,0.7824
1,She works on NLP systems,She builds natural language processing pipelines,Relevant,0.7037
2,Transformers generate embeddings,Sentence embeddings help semantic search,Relevant,0.3638
3,I love machine learning,The weather is hot today,Not Relevant,0.0109
4,He plays football,Natural language processing uses text data,Not Relevant,0.0691
5,Python is a programming language,Snakes live in forests,Not Relevant,0.1461


In [8]:
thresholds = [0.9, 0.75, 0.6, 0.45]

for t in thresholds:
    df[f"Predicted @ {t}"] = df["Similarity"].apply(
        lambda x: "Relevant" if x >= t else "Not Relevant"
    )

df

,Sentence A,Sentence B,Expected,Similarity,Predicted @ 0.9,Predicted @ 0.75,Predicted @ 0.6,Predicted @ 0.45
0,I love machine learning,I enjoy studying machine learning,Relevant,0.7824,Not Relevant,Relevant,Relevant,Relevant
1,She works on NLP systems,She builds natural language processing pipelines,Relevant,0.7037,Not Relevant,Not Relevant,Relevant,Relevant
2,Transformers generate embeddings,Sentence embeddings help semantic search,Relevant,0.3638,Not Relevant,Not Relevant,Not Relevant,Not Relevant
3,I love machine learning,The weather is hot today,Not Relevant,0.0109,Not Relevant,Not Relevant,Not Relevant,Not Relevant
4,He plays football,Natural language processing uses text data,Not Relevant,0.0691,Not Relevant,Not Relevant,Not Relevant,Not Relevant
5,Python is a programming language,Snakes live in forests,Not Relevant,0.1461,Not Relevant,Not Relevant,Not Relevant,Not Relevant


In [ ]:
analysis = []

for t in thresholds:
    correct = (df["Expected"] == df[f"Predicted @ {t}"]).sum()
    total = len(df)
    analysis.append({
        "Threshold": t,
        "Correct Predictions": correct,
        "Accuracy": round(correct / total, 2)
    })

pd.DataFrame(analysis)